In [ ]:
import pathlib
import sys
sys.path.append("..")

import numpy as np
import pickle
import matplotlib.pyplot as plt

import torch

from dataset_configs.smpl.utils import compute_B_from_beta, smpl_pose_to_D_orientation
from dataset_configs.movi import consts as movi_consts
from dataset_configs.movi.utils import (
    load_smpl_params,
    load_imu_data,
    generate_default_placement_params,
    generate_B_range,
)
from pipeline.resample import align_to_smpl_rate
from wimusim import WIMUSim, Optimizer, utils

## Introduction

This notebook demonstrates parameter identification using the **MoVi** dataset.

### Workflow
1. **Load MoVi data** — SMPL fits (60 Hz) + synchronized IMU (100 Hz)
2. **Resample** — align IMU to SMPL rate (60 Hz)
3. **Initialize B, D, P, H** from SMPL parameters
4. **Optimize** — gradient descent to match virtual ↔ real IMU
5. **Save** identified parameters for CPM (`cpm_params_movi.pkl`)

## 1. Load MoVi Data

**MoVi** provides both SMPL parameters and synchronized IMU data in the same recording,
making it ideal for parameter identification:
- SMPL fits (AMASS format `.npz`) → initialize B and D
- 17 Xsens IMUs (100 Hz `.pkl`) → real IMU target data

Set `MOVI_PATH` to the root directory of the downloaded MoVi dataset.

In [ ]:
MOVI_PATH       = "path/to/movi"
SMPL_MODEL_PATH = "path/to/smpl/models"
subject  = "F_Subject01"
activity = "walk"
trial    = 1

# Load raw data
betas, global_orient, body_pose = load_smpl_params(MOVI_PATH, subject, activity, trial)
imu_dict_raw = load_imu_data(MOVI_PATH, subject, activity, trial)

# Align IMU (100 Hz) to SMPL rate (60 Hz) using SLERP + linear interp
global_orient, body_pose, imu_dict, target_hz = align_to_smpl_rate(
    global_orient, body_pose,
    imu_dict=imu_dict_raw,
    smpl_hz=movi_consts.SMPL_SAMPLE_RATE,   # 60 Hz
    imu_hz=movi_consts.IMU_SAMPLE_RATE,     # 100 Hz
)

n_samples = min(v[0].shape[0] for v in imu_dict.values())
print(f"Subject: {subject} | Activity: {activity} | Trial: {trial}")
print(f"After alignment — frames: {n_samples} @ {target_hz} Hz")
print(f"IMUs: {list(imu_dict.keys())}")

## 2. Initialize WIMUSim Parameters

### Step 2.1: Initialize Dynamics (D)
We initialize the **Dynamics (D)** parameters directly from the SMPL pose estimation
output using `smpl_pose_to_D_orientation`. This converts the SMPL rotation matrices to
per-joint quaternions in WXYZ format (WIMUSim convention).

> The video-based SMPL pose runs at a different frame rate (e.g. 30 Hz from HMR2.0)
> than the IMU (50 Hz). The optimization will handle the mismatch; alternatively
> you can resample to match before passing to WIMUSim.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU.")

orientation = smpl_pose_to_D_orientation(global_orient, body_pose)
D_dict = {"orientation": orientation}

D = WIMUSim.Dynamics(
    **D_dict,
    device=device,
    sample_rate=movi_consts.SMPL_SAMPLE_RATE,
    requires_grad=True,
)

### Step 2.2: Initialize Body (B), Placement (P), and Hardware (H)

- **B**: Bone vectors computed from SMPL β using a T-pose forward pass.
- **P**: Default IMU placement offsets derived from the bone vectors.
- **H**: Default hardware noise/bias models.

In [ ]:
smpl_model_path = "path/to/smpl/models"

B_rp       = compute_B_from_beta(betas, smpl_model_path=smpl_model_path, gender="neutral")
B_rp_range = generate_B_range(B_rp)

B = WIMUSim.Body(
    rp=B_rp,
    rp_range_dict=B_rp_range,
    device=device,
    requires_grad=True,
)

P_params = generate_default_placement_params(B_rp)
P = WIMUSim.Placement(
    **P_params,
    device=device,
    requires_grad=True,
)

H = WIMUSim.Hardware(
    **utils.generate_default_H_configs(P.imu_names),
    device=device,
    requires_grad=True,
)

### Step 3: **Launch the WIMUSim Environment**
Finally, we create an instance of the WIMUSim environment using the initialized B, D, P, and H parameters. With this setup, the WIMUSim environment is now you can generate virtual IMU data by running the wimusim_env.simulate() method.

In [ ]:
wimusim_env = WIMUSim(B=B, D=D, P=P, H=H, dataset_name="SMPL")

In [ ]:
# Visualize pre-optimization virtual IMU for LLA (left lower arm)
virtual_IMU_dict = wimusim_env.simulate(mode="generate")

fig, axs = plt.subplots(3, 2, figsize=(15, 6))
acc_LLA_pre_opt, gyro_LLA_pre_opt = virtual_IMU_dict["LLA"]
start, end = 0, 500  # 10 sec at 50 Hz
axs[0, 0].set_title("Virtual Accelerometer (m/s²)")
axs[0, 1].set_title("Virtual Gyroscope (rad/s)")
for i in range(3):
    axs[i, 0].plot(acc_LLA_pre_opt.detach().cpu().numpy()[start:end, i], label=f"Acc{['X', 'Y', 'Z'][i]}")
    axs[i, 0].legend()
    axs[i, 1].plot(gyro_LLA_pre_opt.detach().cpu().numpy()[start:end, i], label=f"Gyro{['X', 'Y', 'Z'][i]}")
    axs[i, 1].legend()
plt.tight_layout()
plt.show()

## 4. Initialize the Optimizer

Next, we set up the optimizer, which is responsible for updating the WIMUSim parameters to minimize the difference between the real and virtual IMU data.


In [ ]:
opt = Optimizer(
    wimusim_env,
    meta_info={"subject": subject, "activity": activity, "trial": trial},
)

### Step 4.1: Set Target IMU Data

Load the real MoVi IMU data (already resampled to 60 Hz) as the optimization target.

In [ ]:
target_IMU_dict = {
    imu_name: (
        torch.tensor(acc[:n_samples],  device=wimusim_env.device),
        torch.tensor(gyro[:n_samples], device=wimusim_env.device),
    )
    for imu_name, (acc, gyro) in imu_dict.items()
    if imu_name in wimusim_env.P.imu_names
}

opt.set_target_IMU_dict(target_IMU_dict)

### Step 4.2: **Define other optimization configurations**

To fine-tune the parameters, we need to define some optimization configurations, including learning rates, loss weights, and scheduling strategies.

In [ ]:
opt.init_optimizers()
opt.scheduler_dict = {
    "Do": torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt.optimizer_dict["Do"], T_0=2, T_mult=2, eta_min=1e-8, verbose=False
    ),
    "B": torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt.optimizer_dict["B"], T_0=2, T_mult=2, eta_min=1e-6, verbose=False
    ),
    "Pp": torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt.optimizer_dict["Pp"], T_0=2, T_mult=2, eta_min=1e-6, verbose=False
    ),
    "Po": torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt.optimizer_dict["Pp"], T_0=2, T_mult=2, eta_min=1e-6, verbose=False
    ),
}

# MoVi 17 IMUs — weight limb-end sensors more heavily
opt.rmse_weight_dict = {
    "HED":  (1.0, 1.0),
    "STER": (1.0, 1.0),
    "PELV": (1.0, 1.0),
    "RSHO": (1.0, 1.0),  "LSHO": (1.0, 1.0),
    "RUA":  (1.0, 2.0),  "LUA":  (1.0, 2.0),
    "RLA":  (2.0, 3.0),  "LLA":  (2.0, 3.0),
    "RHD":  (2.0, 3.0),  "LHD":  (2.0, 3.0),
    "RTH":  (1.0, 2.0),  "LTH":  (1.0, 2.0),
    "RSH":  (2.0, 3.0),  "LSH":  (2.0, 3.0),
    "RFT":  (2.0, 3.0),  "LFT":  (2.0, 3.0),
}

opt.loss_coeff_dict["sym"]     = 1e1
opt.loss_coeff_dict["rom"]     = 1e1
opt.loss_coeff_dict["b_range"] = 1e2
opt.loss_coeff_dict["p_range"] = 1e3

opt.loss_coeff_dict

### Step 5: **Run the Optimization**

Finally, we run the optimization process. During this process, the optimizer will iteratively update the parameters to minimize the error between the real and virtual IMU data. The following code use wandb to log the optimization process, but you can disable it by setting `log_wandb=False`.

In [ ]:
wandb_project_name = "wimusim_movi_param_opt"
rand_int = np.random.randint(1000)
wandb_run_name = f"{subject}_{activity}_t{trial}_{rand_int}"

_ = opt.fit(
    epochs=1024,
    early_stopping=False,
    log_wandb=True,
    wandb_project_config={
        "project_name": wandb_project_name,
        "run_name": wandb_run_name,
    }
)

## Visualization and Analysis

### 1. **Comparing Real and Virtual IMU Data**
To evaluate the effectiveness of the parameter identification process, we will visualize the real IMU data (`target_IMU`), the initial (pre-optimization) virtual IMU data, and the optimized virtual IMU data on the same plot. This comparison allows us to see how closely the virtual IMU data has aligned with the real IMU data after optimization.

The following plot compares the acc and gyro of target IMU (real data), pre-optimization virtual IMU, optimized virtual IMU for RLA IMU:

In [13]:
virtual_IMU_dict_opt = opt.env.simulate(mode="parameterise")

In [ ]:
def compare_real_virtual_IMU(target_IMU_dict, virtual_IMU_dict, imu_name="LLA", start=0, end=500):
    fig, axs = plt.subplots(3, 2, figsize=(15, 6))
    acc_real, gyro_real = target_IMU_dict[imu_name]
    acc_sim,  gyro_sim  = virtual_IMU_dict[imu_name]

    axs[0, 0].set_title(f"Real vs Virtual Accelerometer — {imu_name} (m/s²)")
    axs[0, 1].set_title(f"Real vs Virtual Gyroscope — {imu_name} (rad/s)")
    for i in range(3):
        label = ['X', 'Y', 'Z'][i]
        axs[i, 0].plot(acc_real.detach().cpu().numpy()[start:end, i], label=f"real_{label}")
        axs[i, 0].plot(acc_sim.detach().cpu().numpy()[start:end, i],  label=f"sim_{label}")
        axs[i, 0].legend()
        axs[i, 1].plot(gyro_real.detach().cpu().numpy()[start:end, i], label=f"real_{label}")
        axs[i, 1].plot(gyro_sim.detach().cpu().numpy()[start:end, i],  label=f"sim_{label}")
        axs[i, 1].legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Compare the real and virtual IMU (pre-opt) data for LLA
compare_real_virtual_IMU(target_IMU_dict, virtual_IMU_dict, imu_name="LLA")

In [ ]:
# Compare the real and virtual IMU (post-opt) data for LLA
compare_real_virtual_IMU(target_IMU_dict, virtual_IMU_dict_opt, imu_name="LLA")

### 2. **3D Visualization in PyBullet**
You can also visualize the moving humanoid in the 3D PyBullet environment. This visualization will show the human body model and the IMU placements, making it easy to see the overall movement patterns.

In [17]:
wimusim_env.launch_pybullet_client()
wimusim_env.run_visualization()

startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=2
argv[0] = --unused
argv[1] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=NVIDIA Corporation
GL_RENDERER=NVIDIA GeForce GTX 1080 Ti/PCIe/SSE2
GL_VERSION=3.3.0 NVIDIA 515.105.01
GL_SHADING_LANGUAGE_VERSION=3.30 NVIDIA via Cg compiler
pthread_getconcurrency()=0
Version = 3.3.0 NVIDIA 515.105.01
Vendor = NVIDIA Corporation
Renderer = NVIDIA GeForce GTX 1080 Ti/PCIe/SSE2
b3Printf: Selected demo: Physics Server
startThreads creating 1 threads.
starting thread 0
started thread 0 
MotionThreadFunc thread started
ven = NVIDIA Corporation
ven = NVIDIA Corporation


## Conclusion
By identifying the parameters, WIMUSim can generate high-fidelity virtual IMU data that closely matches real-world recordings.

## 6. Save Identified Parameters for CPM

After identification, save B / D / P / H and the target IMU data so that
`parameter_transformation.ipynb` can load them for data augmentation.

Run this notebook for **each subject/activity** in MoVi and accumulate the lists
before saving — the CPM object mixes parameters across all entries.

In [ ]:
# ── Accumulate across multiple subjects/activities ────────────────────────
# Run the full notebook for each sequence, then append here before saving.
# For a single sequence (demo), the lists have length 1.

cpm_params_path = "cpm_params_movi.pkl"

# Load existing file if it already exists (to append new subjects)
try:
    with open(cpm_params_path, "rb") as f:
        cpm_params = pickle.load(f)
    print(f"Loaded existing file with {len(cpm_params['B_list'])} entries.")
except FileNotFoundError:
    cpm_params = {"B_list": [], "D_list": [], "P_list": [], "H_list": [], "target_list": []}
    print("Creating new CPM params file.")

# Append current subject's optimized parameters
cpm_params["B_list"].append(opt.env.B)
cpm_params["D_list"].append(opt.env.D)
cpm_params["P_list"].append(opt.env.P)
cpm_params["H_list"].append(opt.env.H)
cpm_params["target_list"].append(target_IMU_dict)

with open(cpm_params_path, "wb") as f:
    pickle.dump(cpm_params, f)

print(f"Saved {len(cpm_params['B_list'])} entries to {cpm_params_path}")